In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.font_manager import fontManager, FontProperties


try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

df_path = "/Users/royalmustaches/Documents/Programming/Dano/DANO_ITMO_2026/Datasets/new_data.csv"
df = pd.read_csv(df_path)

PALETTE = ["#5e17eb", "#ff6b6b", "#00c49a"]
FONT_CANDIDATES = [
    "/Users/royalmustaches/Documents/Programming/Dano/DANO_ITMO_2026/HSE_FONTS_FOR_GRAPHS_copy/HSESlab-Regular.ttf",
    "/Users/royalmustaches/Documents/Programming/Dano/DANO_NES_2026/HSE_FONTS_FOR_GRAPHS/HSESlab-Regular.ttf",
    "/Users/royalmustaches/Documents/Programming/Dano/Practice/Organization/Fonts_for_projects/HSESlab-Regular.ttf",
]

FONT_NAME = "DejaVu Sans"
for font_path in FONT_CANDIDATES:
    if os.path.exists(font_path):
        fontManager.addfont(font_path)
        FONT_NAME = FontProperties(fname=font_path).get_name()
        break

sns.set_theme(style="whitegrid", font=FONT_NAME, context="notebook", palette=PALETTE)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["grid.linestyle"] = "--"

PRIMARY = "#5e17eb"
SECONDARY = "#ff6b6b"
ACCENT = "#00c49a"

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =========================
# 1. Подготовка данных
# =========================

df_model = df.copy()

df_model.columns = (
    df_model.columns
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
)

df_model['order_date'] = pd.to_datetime(df_model['order_date'])

df_model = df_model[
    (df_model['gmv_without_markup'] > 0) &
    (df_model['markup'].isin([0, 1, 5, 9]))
].copy()

df_model = df_model.dropna(
    subset=[
        'client_id',
        'order_id',
        'order_date',
        'markup',
        'cb_percent',
        'lifestyle___super_cnt_90',
        'gender_cd',
        'age_group',
        'income_group',
        'mb_events___food_main_cnt_90',
        'mb_events___gorod_cnt_90'
    ]
).copy()

# =========================
# 2. Целевая переменная: повтор в течение 7 дней
# =========================

client_dates = (
    df_model[['client_id', 'order_date']]
    .drop_duplicates()
    .sort_values(['client_id', 'order_date'])
)

client_dates['next_order_date'] = (
    client_dates
    .groupby('client_id')['order_date']
    .shift(-1)
)

df_model = df_model.merge(
    client_dates,
    on=['client_id', 'order_date'],
    how='left'
)

df_model['days_to_next_order'] = (
    df_model['next_order_date'] - df_model['order_date']
).dt.days

df_model['repeat_7d'] = (
    (df_model['days_to_next_order'] >= 1) &
    (df_model['days_to_next_order'] <= 7)
).astype(int)

# Убираем последние 7 дней наблюдения, иначе для них нельзя честно проверить повтор
max_date = df_model['order_date'].max()

df_model = df_model[
    df_model['order_date'] <= max_date - pd.Timedelta(days=7)
].copy()

# =========================
# 3. Признаки
# =========================

df_model['experience_90_log'] = np.log1p(df_model['lifestyle___super_cnt_90'])
df_model['experience_90_log_c'] = (
    df_model['experience_90_log'] - df_model['experience_90_log'].mean()
)

df_model['cb_c'] = df_model['cb_percent'] - df_model['cb_percent'].mean()

df_model['food_main_log_c'] = (
    np.log1p(df_model['mb_events___food_main_cnt_90']) -
    np.log1p(df_model['mb_events___food_main_cnt_90']).mean()
)

df_model['gorod_log_c'] = (
    np.log1p(df_model['mb_events___gorod_cnt_90']) -
    np.log1p(df_model['mb_events___gorod_cnt_90']).mean()
)

# =========================
# 4. Логистическая регрессия
# =========================

formula = """
repeat_7d ~ C(markup) * experience_90_log_c
          + cb_c
          + C(age_group)
          + C(gender_cd)
          + C(income_group)
          + food_main_log_c
          + gorod_log_c
          + C(order_date)
"""

model = smf.glm(
    formula=formula,
    data=df_model,
    family=sm.families.Binomial()
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['client_id']}
)

# =========================
# 5. Ключевые коэффициенты
# =========================

terms = [
    'C(markup)[T.1]',
    'C(markup)[T.5]',
    'C(markup)[T.9]',
    'experience_90_log_c',
    'C(markup)[T.1]:experience_90_log_c',
    'C(markup)[T.5]:experience_90_log_c',
    'C(markup)[T.9]:experience_90_log_c',
    'cb_c'
]

result_table = pd.DataFrame({
    'coef': model.params[terms],
    'odds_ratio': np.exp(model.params[terms]),
    'p_value': model.pvalues[terms],
    'ci_low_or': np.exp(model.conf_int().loc[terms, 0]),
    'ci_high_or': np.exp(model.conf_int().loc[terms, 1])
})

result_table


,coef,odds_ratio,p_value,ci_low_or,ci_high_or
C(markup)[T.1],-0.175432,0.839095,5.479934e-09,0.791053,0.890054
C(markup)[T.5],-0.192454,0.824933,9.416950e-11,0.778256,0.874408
C(markup)[T.9],-0.223682,0.799570,1.186560e-13,0.753686,0.848247
experience_90_log_c,0.980090,2.664696,0.000000e+00,2.535569,2.800399
C(markup)[T.1]:experience_90_log_c,0.019321,1.019509,5.496308e-01,0.956981,1.086123
C(markup)[T.5]:experience_90_log_c,0.073485,1.076252,2.333043e-02,1.010030,1.146816
C(markup)[T.9]:experience_90_log_c,0.069123,1.071568,3.381303e-02,1.005301,1.142203
cb_c,0.010283,1.010336,9.014683e-08,1.006534,1.014152


In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# =========================
# 1. Подготовка данных
# =========================

df_model = df.copy()

df_model.columns = (
    df_model.columns
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
)

df_model['order_date'] = pd.to_datetime(df_model['order_date'])

df_model = df_model[
    (df_model['markup'].isin([0, 1, 5, 9]))
].copy()

df_model = df_model.dropna(
    subset=[
        'client_id',
        'order_date',
        'markup',
        'lifestyle___super_cnt_90'
    ]
).copy()

# =========================
# 2. Целевая переменная: повтор в течение 7 дней
# =========================

client_dates = (
    df_model[['client_id', 'order_date']]
    .drop_duplicates()
    .sort_values(['client_id', 'order_date'])
)

client_dates['next_order_date'] = (
    client_dates
    .groupby('client_id')['order_date']
    .shift(-1)
)

df_model = df_model.merge(
    client_dates,
    on=['client_id', 'order_date'],
    how='left'
)

df_model['days_to_next_order'] = (
    df_model['next_order_date'] - df_model['order_date']
).dt.days

df_model['repeat_7d'] = (
    (df_model['days_to_next_order'] >= 1) &
    (df_model['days_to_next_order'] <= 7)
).astype(int)

# Убираем последние 7 дней наблюдения
max_date = df_model['order_date'].max()

df_model = df_model[
    df_model['order_date'] <= max_date - pd.Timedelta(days=7)
].copy()

# =========================
# 3. Опыт клиента
# =========================

df_model['experience_90_log'] = np.log1p(df_model['lifestyle___super_cnt_90'])

df_model['experience_90_log_c'] = (
    df_model['experience_90_log'] - df_model['experience_90_log'].mean()
)

# =========================
# 4. Логистическая регрессия без лишних предикторов
# =========================

formula = """
repeat_7d ~ C(markup) * experience_90_log_c
"""

model = smf.glm(
    formula=formula,
    data=df_model,
    family=sm.families.Binomial()
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_model['client_id']}
)

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:              repeat_7d   No. Observations:                51650
Model:                            GLM   Df Residuals:                    51642
Model Family:                Binomial   Df Model:                            7
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -31622.
Date:                Sat, 25 Apr 2026   Deviance:                       63243.
Time:                        17:55:15   Pearson chi2:                 5.18e+04
No. Iterations:                     4   Pseudo R-squ. (CS):             0.1382
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

In [14]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# =========================
# 1. Подготовка данных
# =========================

df_model = df.copy()

df_model.columns = (
    df_model.columns
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
)

# Оставляем только нужные переменные
df_model = df_model.dropna(
    subset=[
        'gmv_with_markup',
        'markup',
        'lifestyle___super_sum'
    ]
).copy()

# Оставляем только положительный GMV
df_model = df_model[df_model['gmv_with_markup'] > 0].copy()

# Оставляем только экспериментальные уровни наценки
df_model = df_model[df_model['markup'].isin([0, 1, 5, 9])].copy()

# Делаем markup категориальной переменной
df_model['markup'] = df_model['markup'].astype('category')

# На всякий случай приводим lifestyle___super_sum к числовому типу
df_model['lifestyle___super_sum'] = pd.to_numeric(
    df_model['lifestyle___super_sum'],
    errors='coerce'
)

df_model = df_model.dropna(subset=['lifestyle___super_sum']).copy()

# =========================
# 2. Линейная регрессия
# =========================

model = smf.ols(
    formula='gmv_with_markup ~ C(markup, Treatment(reference=0)) * lifestyle___super_sum',
    data=df_model
).fit(cov_type='HC3')

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        gmv_with_markup   R-squared:                       0.092
Model:                            OLS   Adj. R-squared:                  0.092
Method:                 Least Squares   F-statistic:                     405.8
Date:                Sat, 25 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:13:59   Log-Likelihood:            -8.3802e+05
No. Observations:               95729   AIC:                         1.676e+06
Df Residuals:                   95721   BIC:                         1.676e+06
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                                                                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------

In [15]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# =========================
# 1. Подготовка данных
# =========================

df_model = df.copy()

df_model.columns = (
    df_model.columns
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
)

# Столбцы с количеством товаров по категориям
good_cnt_cols = [
    'done_food_good_cnt',
    'bread_and_cake_good_cnt',
    'fruits_and_vegetables_good_cnt',
    'water_sodas_and_drinks_good_cnt',
    'sweets_good_cnt'
]

# Приводим к числовому типу
for col in good_cnt_cols + ['lifestyle___super_sum']:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

# Создаем целевую переменную: количество товаров в корзине
df_model['basket_good_cnt'] = df_model[good_cnt_cols].sum(axis=1)

# Оставляем нужные наблюдения
df_model = df_model.dropna(
    subset=[
        'basket_good_cnt',
        'markup',
        'lifestyle___super_sum'
    ]
).copy()

# Оставляем только экспериментальные уровни наценки
df_model = df_model[df_model['markup'].isin([0, 1, 5, 9])].copy()

# Убираем некорректные корзины
df_model = df_model[df_model['basket_good_cnt'] > 0].copy()

# Делаем markup категориальной переменной
df_model['markup'] = df_model['markup'].astype('category')

# =========================
# 2. Линейная регрессия
# =========================

model = smf.ols(
    formula='basket_good_cnt ~ C(markup, Treatment(reference=0)) * lifestyle___super_sum',
    data=df_model
).fit(cov_type='HC3')

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        basket_good_cnt   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.064
Method:                 Least Squares   F-statistic:                     274.2
Date:                Sat, 25 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:18:01   Log-Likelihood:            -2.3967e+05
No. Observations:               87343   AIC:                         4.794e+05
Df Residuals:                   87335   BIC:                         4.794e+05
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                                                                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------